# HyperMARL Quickstart

<div align="center">
  <img src="https://raw.githubusercontent.com/kaleabtessera/hypermarl/main/assets/hypermarl.png" alt="HyperMARL" width="500">
  
  [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kaleabtessera/hypermarl/blob/main/quickstart.ipynb)
  [![Paper](https://img.shields.io/badge/arXiv-2412.04233-b31b1b.svg)](https://arxiv.org/abs/2412.04233)
  [![GitHub](https://img.shields.io/badge/GitHub-HyperMARL-blue.svg)](https://github.com/KaleabTessera/HyperMARL)
</div>


Short notebook to get started with HyperMARL, and see how it differs from traditional networks.

### Key Differences:
- **Common Shared Actor/Critic Network**: Agent share the same parameters, conditioned on observations and an agent ID.
- **HyperMARL**: Uses hypernetworks conditioned on agent embeddings (can be an agent ID or learned embeddings) to generate agent-specific parameters for the actor and/or critic networks. These networks only receive the agent's observations as input.

## 0. Imports

In [ ]:
import jax
import jax.numpy as jnp
import flax.linen as nn
import numpy as np
from enum import Enum
from typing import List, Tuple

## 1. Common Shared Actor-Critic Network

A standard AC network where all agents share the same parameters, conditioned on observations and an agent ID.

In [ ]:
from flax.linen.initializers import constant, orthogonal


class ActionType(Enum):
    """Supported action space types."""

    DISCRETE = "discrete"
    CONTINUOUS = "continuous"


class TraditionalActorCritic(nn.Module):
    """
    Actor-Critic network for IPPO with shared weights.

    All agents use the same network parameters, with agent-specific behavior
    arising from agent IDs being part of the observation.
    """

    action_dim: int  # Dimensionality of action space
    action_type: ActionType = "discrete"  # Type of action space (discrete/continuous)
    activation: str = "tanh"  # Activation function ("tanh" or "relu")
    actor_layers: List[int] = (64, 64)  # Hidden layer sizes for actor network
    critic_layers: List[int] = (64, 64)  # Hidden layer sizes for critic network

    @nn.compact
    def __call__(self, x):
        """
        Forward pass through the actor-critic network.

        Args:
            x: Input observation tensor with agent ID

        Returns:
            actor_output: Action distribution parameters
            critic_value: Value function estimate
        """
        # Select activation function
        if self.activation == "relu":
            activation = nn.relu
        else:
            activation = nn.tanh

        def build_network(x, layer_sizes: List[int], activation):
            """Build an MLP with the specified layer sizes and activation."""
            for size in layer_sizes:
                x = nn.Dense(
                    size,
                    kernel_init=orthogonal(np.sqrt(2)),
                    bias_init=constant(0.0),
                )(x)
                x = activation(x)
            return x

        # Actor network
        hidden = build_network(x, self.actor_layers, activation)
        if self.action_type is ActionType.DISCRETE:
            actor_output = nn.Dense(
                self.action_dim, kernel_init=orthogonal(0.01), bias_init=constant(0.0)
            )(hidden)
        else:  # continuous
            actor_mean = nn.Dense(
                self.action_dim, kernel_init=orthogonal(0.01), bias_init=constant(0.0)
            )(hidden)
            actor_logtstd = self.param(
                "log_std", nn.initializers.zeros, (self.action_dim,)
            )
            actor_output = (actor_mean, actor_logtstd)

        # Critic network
        critic = build_network(x, self.critic_layers, activation)
        critic = nn.Dense(1, kernel_init=orthogonal(1.0), bias_init=constant(0.0))(
            critic
        )

        return actor_output, jnp.squeeze(critic, axis=-1)

## 2. HyperMARL

HyperMARL uses hypernetworks conditioned on agent embeddings (can be an agent ID or learned embeddings) to generate agent-specific parameters for the actor and/or critic networks. These networks only receive the agent's observations as input.

In [ ]:
class HyperNetType(Enum):
    ACTOR = 0
    CRITIC = 1

class MLPHyperNetwork(nn.Module):
    """
    MLP HyperNetwork for generating weights and biases of target networks.

    Unlike linear hypernetworks, this uses multi-layer perceptrons to generate
    the parameters, allowing for more complex mappings between agent embeddings
    and policy weights.
    """
    output_dims: List[Tuple[int, int]]  # List of (input_dim, output_dim) for each layer
    hypernet_type: HyperNetType         # Type of hypernetwork (ACTOR or CRITIC)
    init_scale: float = np.sqrt(2)      # Default initialization scale for ReLU
    use_bias: bool = True               # Whether to use bias in the hypernetwork
    hidden_dims: List[int] = (32,)      # Hidden layer sizes for the MLP , 32 works well for most cases

    @staticmethod
    def hypernet_init(gain, fan_in, fan_out):
        """
        Custom initialization function for the hypernetwork weights.
        Simply takes an initialization function and applies it to each agent's generated weights e.g. orthogonal init weights are commonly used, so this ensures that this generates orthogonal weights for each agent at init.

        Args:
            gain: Scaling factor for weights
            fan_in: Input dimension
            fan_out: Output dimension

        Returns:
            Initialization function
        """
        def weight_init(key, shape, dtype):
            # orthogonal is just to match the init in common actor-critic networks (included in traditional ac networks as well)
            init = jax.nn.initializers.orthogonal(gain)
            batched_init = jax.vmap(init, in_axes=(0, None, None))

            # Create a batch of keys (one per agent)
            batch_size = shape[0]
            keys = jax.random.split(key, num=batch_size)

            # Generate orthogonal weights for each agent
            weights = batched_init(keys, (fan_in, fan_out), dtype)
            return weights.reshape(shape)
        return weight_init

    @nn.compact
    def __call__(self, x):
        """
        Generate weights and biases for each layer of the target network.

        Args:
            x: Agent embeddings

        Returns:
            weight_heads: Generated weights for each layer
            bias_heads: Generated biases for each layer
        """
        weight_heads = []
        bias_heads = []
        for i, (input_dim, output_dim) in enumerate(self.output_dims):
            weight_dim = input_dim * output_dim
            bias_dim = output_dim

            is_final_layer = i == len(self.output_dims) - 1

            # Determine gain for initialization based on layer type and position
            if is_final_layer and self.hypernet_type == HyperNetType.ACTOR:
                gain = 0.01  # Lower gain for final actor layer (common in PPO)
            elif is_final_layer and self.hypernet_type == HyperNetType.CRITIC:
                gain = 1.0   # Standard gain for critic output
            else:
                gain = self.init_scale  # Hidden layer gain

            # MLP for weights generation
            weight_mlp = x
            for hidden_dim in self.hidden_dims:
                weight_mlp = nn.Dense(hidden_dim, use_bias=self.use_bias)(weight_mlp)
                weight_mlp = nn.relu(weight_mlp)
            weight_head = nn.Dense(
                weight_dim,
                use_bias=self.use_bias,
                kernel_init=self.hypernet_init(gain, input_dim, output_dim),
                bias_init=nn.initializers.zeros,
            )(weight_mlp)

            # MLP for biases generation
            bias_mlp = x
            for hidden_dim in self.hidden_dims:
                bias_mlp = nn.Dense(hidden_dim, use_bias=self.use_bias)(bias_mlp)
                bias_mlp = nn.relu(bias_mlp)
            bias_head = nn.Dense(
                bias_dim,
                use_bias=self.use_bias,
                kernel_init=nn.initializers.zeros,
                bias_init=nn.initializers.zeros,
            )(bias_mlp)

            weight_heads.append(weight_head)
            bias_heads.append(bias_head)

        return weight_heads, bias_heads

In [ ]:
class HyperMARLActorCritic(nn.Module):
    """
    ActorCritic network using MLP hypernetworks to generate agent-specific parameters.

    The architecture uses MLP hypernetworks to generate weights for both the actor
    and critic networks from agent embeddings, allowing parameter sharing while
    supporting complex agent-specific specialization.
    """

    action_dim: int  # Dimension of the action space
    num_agents: int  # Number of agents in the environment
    actor_layers: List[int]  # Hidden layer sizes for the actor network
    critic_layers: List[int]  # Hidden layer sizes for the critic network
    observation_dim: int  # Dimension of the observation space
    embedding_dim: int = 4  # Dimension of the agent embeddings
    init_scale: float = np.sqrt(2)  # Scale for weight initialization
    activation: str = "tanh"  # Activation function to use
    use_agent_id_embeddings: bool = False  # Whether to use learned agent ID embeddings
    use_bias_in_hypernet: bool = False  # Whether to use bias in the hypernetwork
    is_continuous: bool = False  # Whether the action space is continuous
    hypernet_hidden_dims: List[int] = (64,)  # Hidden layer sizes for MLP hypernetworks

    def setup(self):
        """Initialize model components."""
        # Set up activation function
        self.activation_fn = jax.nn.relu if self.activation == "relu" else jax.nn.tanh

        # Initialize agent embeddings - either learned or one-hot
        self.agent_embeddings = (
            self.param(
                "agent_embeddings",
                nn.initializers.orthogonal(self.init_scale),
                (self.num_agents, self.embedding_dim),
            )
            if self.use_agent_id_embeddings
            else jnp.eye(self.num_agents)
        )

        # Compute output dimensions for actor and critic networks
        self.actor_output_dims = self._compute_output_dims(
            self.actor_layers, self.action_dim
        )
        self.critic_output_dims = self._compute_output_dims(self.critic_layers, 1)

        # Initialize hypernetworks for actor and critic
        self.actor_hypernet = MLPHyperNetwork(
            output_dims=self.actor_output_dims,
            init_scale=self.init_scale,
            use_bias=self.use_bias_in_hypernet,
            hypernet_type=HyperNetType.ACTOR,
            hidden_dims=self.hypernet_hidden_dims,
        )
        self.critic_hypernet = MLPHyperNetwork(
            output_dims=self.critic_output_dims,
            init_scale=self.init_scale,
            use_bias=self.use_bias_in_hypernet,
            hypernet_type=HyperNetType.CRITIC,
            hidden_dims=self.hypernet_hidden_dims,
        )

        # Initialize log_std parameter for continuous actions
        if self.is_continuous:
            self.log_std = self.param(
                "log_std", nn.initializers.zeros, (self.action_dim,)
            )

    def _compute_output_dims(self, layers, final_dim):
        """
        Compute the dimensions of each layer in the network.

        Args:
            layers: List of hidden layer sizes
            final_dim: Output dimension of the final layer

        Returns:
            List of (input_dim, output_dim) tuples for each layer
        """
        output_dims = []
        input_dim = self.observation_dim
        for layer_size in layers + (final_dim,):
            output_dims.append((input_dim, layer_size))
            input_dim = layer_size
        return output_dims

    @nn.compact
    def __call__(self, x):
        """
        Forward pass through the actor-critic network.

        Args:
            x: Input tensor containing observation and agent ID

        Returns:
            actor_outputs: Policy distribution parameters
            critic_values: Value function estimates
        """
        # Split input into observation and agent ID
        obs = x[..., : -self.num_agents]
        agent_id = jnp.argmax(x[..., -self.num_agents :], axis=-1)

        actor_outputs, critic_outputs = self._apply_networks(obs, agent_id)
        return actor_outputs, jnp.squeeze(critic_outputs, axis=-1)

    def _apply_networks(self, obs, agent_id):
        """
        Apply the hypernetworks to generate actor and critic outputs.

        Args:
            obs: Observation tensor
            agent_id: Agent IDs

        Returns:
            actor_outputs: Policy distribution parameters
            critic_outputs: Value function estimates
        """
        batch_size = obs.shape[0]

        # Pre-compute all hypernet outputs for all agents
        actor_weights, actor_biases = self.actor_hypernet(self.agent_embeddings)
        critic_weights, critic_biases = self.critic_hypernet(self.agent_embeddings)

        def apply_hypernet(obs, weights, biases):
            """
            Apply hypernetwork-generated weights to an input.

            Args:
                obs: Observation tensor
                weights: List of weight matrices
                biases: List of bias vectors

            Returns:
                Network output
            """
            x = obs
            # Apply all hidden layers with activation
            for w, b in zip(weights[:-1], biases[:-1]):
                x = self.activation_fn(jnp.matmul(x, w.reshape(x.shape[-1], -1)) + b)
            # Apply final layer without activation
            return jnp.matmul(x, weights[-1].reshape(x.shape[-1], -1)) + biases[-1]

        # Vectorize apply_hypernet over all agents
        vmap_apply_hypernet = jax.vmap(apply_hypernet, in_axes=(None, 0, 0))

        # Apply networks for all agents simultaneously
        # Shape: (num_agents, batch_size, action_dim) for actor
        # Shape: (num_agents, batch_size, 1) for critic
        actor_outputs_all = vmap_apply_hypernet(obs, actor_weights, actor_biases)
        critic_outputs_all = vmap_apply_hypernet(obs, critic_weights, critic_biases)

        # Select outputs for the specific agents in the batch
        batch_indices = jnp.arange(batch_size)
        actor_outputs = actor_outputs_all[agent_id, batch_indices]
        critic_outputs = critic_outputs_all[agent_id, batch_indices]

        # For continuous action spaces, return mean and log_std
        if self.is_continuous:
            return (actor_outputs, self.log_std), critic_outputs
        else:
            return actor_outputs, critic_outputs

## 3. Comparison Setup

In [ ]:
# Helper functions
def create_dummy_input(batch_size, observation_dim, num_agents, seed=42):
    """Create dummy input: [observation, one_hot_agent_id]"""
    key = jax.random.PRNGKey(seed)

    # Random observations
    obs = jax.random.normal(key, (batch_size, observation_dim))

    # Random agent IDs
    key, subkey = jax.random.split(key)
    agent_ids = jax.random.randint(subkey, (batch_size,), 0, num_agents)
    one_hot_agents = jax.nn.one_hot(agent_ids, num_agents)

    return jnp.concatenate([obs, one_hot_agents], axis=-1)

def summarize_output(output, name):
    """Print summary statistics of network output."""
    output_np = np.array(output)
    print(f"{name}:")
    print(f"  Shape: {output_np.shape}")
    print(f"  Mean: {np.mean(output_np):.4f}")
    print(f"  Std: {np.std(output_np):.4f}")
    print(f"  Range: [{np.min(output_np):.4f}, {np.max(output_np):.4f}]")

## 4. Initialise and Forward Pass

In [ ]:
# Configuration
batch_size = 100
observation_dim = 10
num_agents = 3
action_dim = 5

# Create dummy input
dummy_input = create_dummy_input(batch_size, observation_dim, num_agents)
print(f"Input shape: {dummy_input.shape}")
print(f"Input breakdown: {observation_dim} obs + {num_agents} agent_id = {dummy_input.shape[-1]}")

# Initialize models
traditional_model = TraditionalActorCritic(action_dim=action_dim, actor_layers=(64, 64), critic_layers=(64, 64), action_type=ActionType.DISCRETE)
hypermarl_model = HyperMARLActorCritic(
    action_dim=action_dim,
    num_agents=num_agents,
    observation_dim=observation_dim,
    use_agent_id_embeddings=False,  # Use one-hot agent IDs
    actor_layers=(64, 64),
    critic_layers=(64, 64),
)

# Initialize parameters
seed = jax.random.PRNGKey(0)
traditional_params = traditional_model.init(seed, dummy_input)
hypermarl_params = hypermarl_model.init(seed, dummy_input)

# Forward pass
traditional_actor, traditional_critic = traditional_model.apply(traditional_params, dummy_input)
hypermarl_actor, hypermarl_critic = hypermarl_model.apply(hypermarl_params, dummy_input)

print("Network Outputs:")
print("="*50)

print("\nTraditional Actor-Critic:")
summarize_output(traditional_actor, "Actor output")
summarize_output(traditional_critic, "Critic output")
print("Shapes", traditional_actor.shape, traditional_critic.shape)

print("\nHyperMARL Actor-Critic:")
summarize_output(hypermarl_actor, "Actor output")
summarize_output(hypermarl_critic, "Critic output")
print("Shapes", hypermarl_actor.shape, hypermarl_critic.shape)

## 5. If you want to tune HyperMARL to your problem, try the following:
- `embedding_dim`: For more complex problems, increase the embedding dimension (e.g., 64 or 128).
- `hypernet_hidden_dim`: Keeping this consistent at 32 worked well for most cases.
- `learning_rate`: hypernetworks generate parameters, which can have different learning dynamics, so you should tune lr and possibly also max grad norm (typically lower values than standard AC).

For more details, refer to the [HyperMARL paper](https://arxiv.org/abs/2412.04233) and the [HyperMARL repository](https://github.com/KaleabTessera/HyperMARL).